# Estimate langmuir crb structured mc

In [ ]:
from __future__ import annotations
import argparse
import json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.sparse.linalg import LinearOperator, cg
from scipy.special import ndtr
from binding_helpers import load_dataset_config, simulate_thickness_traces
from stochastic_fano import load_stochastic_fano, sample_spectral_images, noise_metadata


In [ ]:
PACKAGE = Path.cwd().parent
ROOT = PROJECT = PROJECT_ROOT = PACKAGE / 'work'
SETTINGS = argparse.Namespace(fano_dir=PACKAGE / 'work/fano_forward_model_double_ellipse_S843_theta205_0to6nm_parameter_noise0p05', dataset_summary=PACKAGE / 'work/binding/protein_langmuir_dataset_summary.csv', out_dir=PACKAGE / 'work/crb/spectrum0p01_seed131', theta_min=-10.5, theta_max=-8.0, n_sections=8, n_realizations=1000, derivative_step=0.025, time_stop_s=7200, time_stride=60, wavelength_stride=20, spectrum_noise_std=0.01, seed=131, cg_rtol=1e-05, cg_maxiter=300, variance_floor=1e-08)


In [ ]:
def smooth_expected_image_mean(theta, args, config, fano, time_s):
    """Mean image at the exact expected occupancy, averaged over Fano noise."""
    concentration = 10.0 ** theta
    dt = float(config['dtSeconds'])
    p_on = min(float(config['kon_Minv_sinv']) * concentration * dt, 1.0)
    p_off = min(float(config['koff_sinv']) * dt, 1.0)
    probability = 0.0
    output_lookup = {int(t): index for index, t in enumerate(time_s)}
    expected_coverage = np.zeros(len(time_s), dtype=np.float64)
    n_steps = int(round(args.time_stop_s / dt))
    for step in range(1, n_steps + 1):
        probability += (1.0 - probability) * p_on - probability * p_off
        current_time = int(round(step * dt))
        if current_time in output_lookup:
            expected_coverage[output_lookup[current_time]] = probability
    scale = np.pi * float(config['proteinDiameterNm']) / 6.0
    thickness = np.broadcast_to((scale * expected_coverage)[None, :], (args.n_realizations, len(time_s))).copy()
    rng = np.random.default_rng(args.seed + 104729)
    latent = sample_spectral_images(thickness, fano, rng).reshape(args.n_realizations, -1)
    conditional_mean, _ = clipped_normal_moments(latent, args.spectrum_noise_std)
    return conditional_mean.mean(axis=0, dtype=np.float64)


In [ ]:
def clipped_normal_moments(location: np.ndarray, sigma: float) -> tuple[np.ndarray, np.ndarray]:
    """Moments of clip(N(location, sigma**2), 0, 1)."""
    if sigma <= 0:
        clipped = np.clip(location, 0.0, 1.0)
        return (clipped, np.zeros_like(clipped))
    a = -location / sigma
    b = (1.0 - location) / sigma
    phi_a = np.exp(-0.5 * a * a) / np.sqrt(2.0 * np.pi)
    phi_b = np.exp(-0.5 * b * b) / np.sqrt(2.0 * np.pi)
    probability = ndtr(b) - ndtr(a)
    upper_mass = 1.0 - ndtr(b)
    mean = location * probability + sigma * (phi_a - phi_b) + upper_mass
    second = (location * location + sigma * sigma) * probability + 2.0 * location * sigma * (phi_a - phi_b) + sigma * sigma * (a * phi_a - b * phi_b) + upper_mass
    variance = np.maximum(second - mean * mean, 0.0)
    return (mean.astype(np.float32), variance.astype(np.float32))


In [ ]:
def structured_mean_fisher(derivative, centred_samples, diagonal, rtol, maxiter):
    if torch.cuda.is_available():
        return woodbury_fisher_cuda(derivative, centred_samples, diagonal)
    x = centred_samples.astype(np.float64, copy=False)
    scale = 1.0 / (len(x) - 1)
    dimension = derivative.size

    def covariance_product(vector):
        return diagonal * vector + x.T @ (x @ vector) * scale
    covariance = LinearOperator((dimension, dimension), matvec=covariance_product, dtype=np.float64)
    covariance_diagonal = diagonal + np.sum(x * x, axis=0) * scale
    preconditioner = LinearOperator((dimension, dimension), matvec=lambda vector: vector / covariance_diagonal, dtype=np.float64)
    count = [0]
    solution, info = cg(covariance, derivative, M=preconditioner, rtol=rtol, atol=0.0, maxiter=maxiter, callback=lambda _: count.__setitem__(0, count[0] + 1))
    residual = np.linalg.norm(covariance_product(solution) - derivative) / np.linalg.norm(derivative)
    fisher = float(derivative @ solution)
    return (fisher, count[0], float(residual))


In [ ]:
def woodbury_fisher_cuda(derivative, centred_samples, diagonal):
    """Exact inverse quadratic for diag(A) + X.T@X/(n-1)."""
    solver = WoodburyCovarianceCUDA(centred_samples, diagonal)
    d = torch.as_tensor(derivative, dtype=solver.dtype, device=solver.device)
    inverse_d = solver.solve(d)
    fisher = float(torch.dot(d.double(), inverse_d.double()).cpu())
    residual = solver.relative_residual(inverse_d, d)
    return (fisher, 1, residual)


In [ ]:
class WoodburyCovarianceCUDA:

    def __init__(self, centred_samples, diagonal):
        self.device = torch.device('cuda')
        self.dtype = torch.float64
        self.x = torch.as_tensor(centred_samples, dtype=self.dtype, device=self.device)
        self.a = torch.as_tensor(diagonal, dtype=self.dtype, device=self.device)
        self.root_scale = float(np.sqrt(len(centred_samples) - 1))
        weighted_x = self.x * torch.rsqrt(self.a)[None, :] / self.root_scale
        kernel = weighted_x @ weighted_x.T
        kernel.diagonal().add_(1.0)
        self.factor = torch.linalg.cholesky(kernel)

    def solve(self, value):
        was_vector = value.ndim == 1
        matrix = value[:, None] if was_vector else value
        inverse_diagonal_matrix = matrix / self.a[:, None]
        rhs = self.x @ inverse_diagonal_matrix / self.root_scale
        coefficient = torch.cholesky_solve(rhs, self.factor)
        result = inverse_diagonal_matrix - self.x.T @ coefficient / (self.root_scale * self.a[:, None])
        return result[:, 0] if was_vector else result

    def multiply(self, value):
        was_vector = value.ndim == 1
        matrix = value[:, None] if was_vector else value
        result = self.a[:, None] * matrix + self.x.T @ (self.x @ matrix) / (len(self.x) - 1)
        return result[:, 0] if was_vector else result

    def relative_residual(self, solution, target):
        return float((torch.linalg.vector_norm(self.multiply(solution) - target) / torch.linalg.vector_norm(target)).cpu())


In [ ]:
def serializable_args(args):
    return {key: str(value) if isinstance(value, Path) else value for key, value in vars(args).items()}


In [ ]:
args = SETTINGS
args.out_dir.mkdir(parents=True, exist_ok=True)
config = load_dataset_config(args.dataset_summary)
fano = load_stochastic_fano(args.fano_dir, args.wavelength_stride)
time_s = np.arange(0, args.time_stop_s + 1, args.time_stride, dtype=np.int32)
edges = np.linspace(args.theta_min, args.theta_max, args.n_sections + 1)
theta = (edges[:-1] + edges[1:]) / 2.0
rows = []


In [ ]:
for index, value in enumerate(theta):
    theta = value
    keep_samples = True

    rng = np.random.default_rng(args.seed)
    thickness = simulate_thickness_traces(concentration=10.0 ** theta, n_realizations=args.n_realizations, time_stop_s=args.time_stop_s, output_time_s=time_s, dt=float(config['dtSeconds']), kon=float(config['kon_Minv_sinv']), koff=float(config['koff_sinv']), n_sites=int(config['nBindingSites']), protein_diameter_nm=float(config['proteinDiameterNm']), rng=rng)
    latent = sample_spectral_images(thickness, fano, rng).reshape(args.n_realizations, -1)

    conditional_mean, conditional_variance = clipped_normal_moments(latent, args.spectrum_noise_std)
    result = {'mean': conditional_mean.mean(axis=0, dtype=np.float64), 'mean_conditional_variance': conditional_variance.mean(axis=0, dtype=np.float64)}
    if keep_samples:
        result['conditional_mean_samples'] = conditional_mean.astype(np.float32)

    centre = result
    minus_mean = smooth_expected_image_mean(value - args.derivative_step, args, config, fano, time_s)
    plus_mean = smooth_expected_image_mean(value + args.derivative_step, args, config, fano, time_s)
    derivative = ((plus_mean - minus_mean) / (2.0 * args.derivative_step)).astype(np.float64)
    centred_samples = centre['conditional_mean_samples']
    centred_result = centred_samples.astype(np.float32, copy=True)
    centred_result -= centred_result.mean(axis=0, keepdims=True)
    samples = centred_result
    diagonal = np.maximum(centre['mean_conditional_variance'], args.variance_floor).astype(np.float64)
    mean_fisher, iterations, residual = structured_mean_fisher(derivative, samples, diagonal, args.cg_rtol, args.cg_maxiter)
    covariance_fisher = 0.0
    fisher = mean_fisher + covariance_fisher
    total_pixel_variance = diagonal + samples.astype(np.float64).var(axis=0, ddof=1)
    diagonal_fisher = float(np.sum(derivative ** 2 / total_pixel_variance))
    rows.append({'section': index + 1, 'log10_lower_M': edges[index], 'log10_upper_M': edges[index + 1], 'theta_midpoint_log10_M': value, 'concentration_midpoint_M': 10.0 ** value, 'structured_fisher_information': fisher, 'mean_derivative_fisher_information': mean_fisher, 'covariance_derivative_fisher_information': covariance_fisher, 'structured_crb_rmse_log10': 1.0 / np.sqrt(fisher), 'same_samples_diagonal_crb_rmse_log10': 1.0 / np.sqrt(diagonal_fisher), 'diagonal_mean_to_structured_mean_fisher_ratio': diagonal_fisher / mean_fisher, 'cg_iterations': iterations, 'cg_relative_residual': residual})
    print(json.dumps(rows[-1]), flush=True)


In [ ]:
results = pd.DataFrame(rows)
results.to_csv(args.out_dir / 'structured_mc_crb_by_concentration_section.csv', index=False)
summary = {'args': serializable_args(args), 'dataset_config': config, 'parameter_noise': noise_metadata(fano), 'observation_dimension': int(len(fano['wavelength_nm']) * len(time_s)), 'covariance_model': 'full Monte Carlo covariance of stochastic binding and shared-wavelength Fano noise plus analytic diagonal conditional variance of clipped Gaussian pixel noise', 'fisher_terms': 'structured Gaussian mean-derivative term; covariance-derivative term is excluded by default because high-dimensional sample-covariance differences are unstable at the available Monte Carlo sample size', 'mean_derivative': 'central difference of the exact expected binding trajectory with common continuous Fano-noise samples; avoids derivatives of discrete binomial draws', 'results': rows}
(args.out_dir / 'structured_mc_crb_summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')